# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [84]:
phase_3_action_plan = {
    "1. Demographic Redundancy": {
        "Finding": "Nationality, International status, and Special Needs show near-zero variance.",
        "Action": "Remove these features to reduce noise and model complexity."
    },
    "2. High-Cardinality Categorical Data": {
        "Finding": "Application mode and Occupations have too many categories (outliers in the 'rare' entry methods).",
        "Action": "Use Target Encoding to map these categories to the mean dropout rate per class."
    },
    "3. Financial & Academic Redundancy": {
        "Finding": "Parental qualifications/occupations and 1st/2nd semester credit data are highly correlated.",
        "Action": "Engineer composite features (e.g., 'Financial_Risk_Score') and drop redundant 2nd sem admin columns."
    },
    "4. Skewed Continuous Variables": {
        "Finding": "Age at enrollment is heavily right-skewed; Curricular units have extreme outlier tails (exam participation).",
        "Action": "Log-transform Age; apply RobustScaler to Curricular units to minimize outlier bias."
    },
    "5. The 'Zero' Signal": {
        "Finding": "0.0 grades in Curricular units are not just 'low'—they represent a distinct behavioral state (Dropout spike).",
        "Action": "Create a 'is_zero_sem1' binary flag to preserve this high-importance signal."
    },
    "6. Data Integrity Verification": {
        "Observation": "Systematic audit confirmed 0 missing values and 0 duplicate records across the entire dataset.",
        "Technical Insight": "The dataset is characterized by 100% completeness and perfect record independence.",
        "Implemented Action": "No corrective measures (Imputation or Deduplication) are required. The pipeline will proceed directly to feature transformation, ensuring 100% signal retention for Phase 3."
    }
}

In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import LabelEncoder

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [86]:
DATA_PATH = r'..\data\raw\datasetOne.csv'
df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded dataset: 4424 rows x 35 columns


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,8,5,2,1,1,1,13,10,6,10,1,0,0,1,1,0,20,0,0,0,0,0,0.000000,0,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,6,1,11,1,1,1,1,3,4,4,1,0,0,0,1,0,19,0,0,6,6,6,14.000000,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,5,1,1,1,22,27,10,10,1,0,0,0,1,0,19,0,0,6,0,0,0.000000,0,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,8,2,15,1,1,1,23,27,6,4,1,0,0,1,0,0,20,0,0,6,8,6,13.428571,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,12,1,3,0,1,1,22,28,10,10,0,0,0,1,0,0,45,0,0,6,9,5,12.333333,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [87]:
# Relevant Columns
columns_to_keep = [
    'Marital status', 'Application mode', 'Application order', 'Course', 
    'Daytime/evening attendance', 'Previous qualification', 'Displaced', 
    'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 
    'Age at enrollment', 'Curricular units 1st sem (enrolled)', 
    'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 
    'Curricular units 1st sem (grade)', 'Target', 'Curricular units 2nd sem (grade)', 
    'Curricular units 2nd sem (approved)',
]

# Columns that will be dropped
columns_to_drop = [
    'Nationality', 'International', 'Educational special needs', # Zero/Near-zero variance
    'Curricular units 1st sem (without evaluations)',            # Zero variance
    'Curricular units 2nd sem (without evaluations)',            # Zero variance
    'Unemployment rate', 'Inflation rate', 'GDP',                 # Noise (Zero correlation to Target)
    'Curricular units 2nd sem (enrolled)',                        # Redundant (0.94 corr with Sem 1)
    'Curricular units 2nd sem (credited)'                         # Redundant (0.94 corr with Sem 1)
]

drop_reason = {
    "Demographics": {
        "Insight": "Nationality, International, and Special Needs showed near-zero variance; providing no signal.",
        "Evidence_Graph": "Count Plots (Bar Charts) — Showed 98%+ of students belonged to a single category, meaning the model can't 'learn' anything from the rare cases."
    },
    "Macro-Economics": {
        "Insight": "Unemployment, Inflation, and GDP showed no statistical correlation with academic success in EDA.",
        "Evidence_Graph": "Correlation Heatmap — These features had coefficients near 0.0 with the 'Target' variable, indicating they are just 'noise' in this specific dataset."
    },
    "Admin Redundancy": {
        "Insight": "2nd Sem Enrolled/Credited are >0.90 correlated with 1st Sem; keeping them causes multicollinearity.",
        "Evidence_Graph": "Correlation Matrix & Scatter Plots — Showed a near-perfect diagonal line between Sem 1 and Sem 2 counts."
    },
    "Zero Variance": {
        "Insight": "Units 'without evaluations' were 0 for almost the entire dataset.",
        "Evidence_Graph": "Histograms — Showed a single massive spike at 0.0 with no spread (standard deviation), making the column mathematically useless for splitting data."
    }
}

In [88]:
# Optional: Filter rows based on specific criteria
# Example: Remove rows where a critical field is missing or filter by a condition

# The dataset was already verified to have no missing values, so I can skip that step.

---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [89]:
# 1. Handle missing values 

# isnull().sum() gives the count of missing values per column, and sum() on that result gives the total count of missing values in  the DF.
missing_count = df.isnull().sum().sum()

if missing_count == 0:
    print("Dataset is already clean. No missing values detected.")
else:
    numerical_cols = df.select_dtypes(include=np.number).columns
    df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].median())
    print(f"Handled {missing_count} unexpected missing values.")


Dataset is already clean. No missing values detected.


In [90]:
# 2. Finding & fixing any inaccurate or corrupted data entries (Data Integrity Audit)

# ---------------1. Logical Range Check------------------------------------
# Flagging grades outside the standard 0-20 scale
invalid_grades = df[(df['Curricular units 1st sem (grade)'] > 20) | (df['Curricular units 2nd sem (grade)'] > 20)]

# Flagging unrealistic age (e.g., younger than 16 or older than 80)
invalid_age = df[(df['Age at enrollment'] < 16) | (df['Age at enrollment'] > 80)]


# ---------------2. Cross-Column Validation------------------
# Graduate with 0 units approved or credited in the final year (conflict: How can a student graduate without any approved or credited units?)
grad_conflict = df[(df['Target'] == 'Graduate') & (df['Curricular units 2nd sem (approved)'] == 0) & (df['Curricular units 2nd sem (credited)'] == 0) & (df['Curricular units 1st sem (approved)'] == 0) & (df['Curricular units 1st sem (credited)'] == 0)]

# Students with grades but 0 evaluations (Conflict: How did they get a grade?)
evaluation_conflict = df[(df['Curricular units 1st sem (evaluations)'] == 0) & (df['Curricular units 1st sem (grade)'] > 0)]

# Approved units exceed Enrolled units (Impossible)
enrollment_conflict = df[df['Curricular units 1st sem (approved)'] > df['Curricular units 1st sem (enrolled)']]

# Summary of issues found
print(f"1. Impossible Grades Found: {len(invalid_grades)}")
print(f"2. Extreme Age Outliers: {len(invalid_age)}")
print(f"3. Logic Conflict (Graduated with 0 Approved Units): {len(grad_conflict)}")
print(f"4. Logic Conflict (Grade with 0 Evaluations): {len(evaluation_conflict)}")
print(f"5. Logic Conflict (Approved > Enrolled): {len(enrollment_conflict)}")
    

1. Impossible Grades Found: 0
2. Extreme Age Outliers: 0
3. Logic Conflict (Graduated with 0 Approved Units): 75
4. Logic Conflict (Grade with 0 Evaluations): 0
5. Logic Conflict (Approved > Enrolled): 0


In [91]:
df_clean = df.copy()

# Distribution pre-drop
binary_df = df_clean[df_clean['Target'] != 'Enrolled'] # Exclude 'Enrolled' class
binary_counts = binary_df['Target'].value_counts() # Calculate counts
binary_pct = binary_df['Target'].value_counts(normalize=True) * 100 # Calculate percentage 
final_distribution = pd.DataFrame({
    'Count': binary_counts,
    'Percentage (%)': binary_pct
})

print("Final Percentage Distribution Before Drop:")
print(final_distribution)

# The drop
df_clean = df_clean.drop(grad_conflict.index)

# Distribution post-drop
binary_df = df_clean[df_clean['Target'] != 'Enrolled']
binary_counts = binary_df['Target'].value_counts()
binary_pct = binary_df['Target'].value_counts(normalize=True) * 100
outcome_report = pd.DataFrame({
    'Count': binary_counts,
    'Percentage (%)': binary_pct
})

print("Final Percentage Distribution After Drop:")
print(outcome_report)

Final Percentage Distribution Before Drop:
          Count  Percentage (%)
Target                         
Graduate   2209       60.853994
Dropout    1421       39.146006
Final Percentage Distribution After Drop:
          Count  Percentage (%)
Target                         
Graduate   2134       60.028129
Dropout    1421       39.971871


In [92]:
cleaning_summary = {
    "action": "Targeted row removal of 75 anomalies.",
    "rationale": "Records labeled as 'Graduate' were excluded due to a total lack of approved or credited units, representing administrative noise that contradicts graduation logic.",
    "impact on distribution": "final_ratio is 60.0% Graduate / 40.0% Dropout, which is a very small baseline_shift of 0.9%.",
    "modeling implication": "Ensures high-fidelity signal and robust decision boundary (generalizes well, not confused by outliers or noise, and stable)."
}

for key, value in cleaning_summary.items():
    print(f"{key}: {value}")


action: Targeted row removal of 75 anomalies.
rationale: Records labeled as 'Graduate' were excluded due to a total lack of approved or credited units, representing administrative noise that contradicts graduation logic.
impact on distribution: final_ratio is 60.0% Graduate / 40.0% Dropout, which is a very small baseline_shift of 0.9%.
modeling implication: Ensures high-fidelity signal and robust decision boundary (generalizes well, not confused by outliers or noise, and stable).


In [93]:
# 3. Remove duplicate records.

before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows.")
print(f"Remaining: {after} rows.")

Removed 0 duplicate rows.
Remaining: 4424 rows.


In [94]:
# 4. Outlier Capping & Log Transformation

# Numerical Columns for Capping (based on boxplots)
academic_cols = [
    'Curricular units 1st sem (enrolled)', 
    'Curricular units 1st sem (evaluations)', 
    'Curricular units 1st sem (approved)'
]

# Function to cap outliers using the IQR method
def cap_outliers_iqr(dataframe, column):

    # Identify the middle 50% of the data (25th and 75th percentiles)
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    
    # Calculate the Interquartile Range
    IQR = Q3 - Q1
    
    # Define values outside these ranges (statistical outliers)
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Force extreme values to sit exactly on the fences (removes outliers)
    dataframe[column] = dataframe[column].clip(lower=lower_bound, upper=upper_bound)
    
    return dataframe

# Apply Capping to Academic features
for col in academic_cols:
    df_clean = cap_outliers_iqr(df_clean, col)

1. Enrolled & Evaluations Semester 1 Units:
Evaluation: A student enrolled in 25 units is likely a data error or a very rare edge case (a typical semester is 5–7 units).
Action: IQR capping will "squish" the 25s and 20s down to a more reasonable maximum (likely around 10–12), which prevents distance-based models from over-weighting these extreme cases.

2. Approved Semester 1 Units:
Since enrolled was capped, Approved must be capped at the same level as to not create a logical impossibility where a student "passed more classes than they were enrolled in."

In [95]:
# Log Transformation for Age
# It applies the formula log(1 + x) to the feature. The +1 ensures the math remains stable if the data contains zeros.
df_clean['Age_at_Enrollment_Log'] = np.log1p(df_clean['Age at enrollment'])

Reason for this change:

1. Fixes Skewness: Age data is "right-skewed" (many young students, few older ones). This pulls the tail in, making the distribution more symmetrical.

2. Stabilizes Variance: It shrinks the mathematical gap between extreme ages (e.g., 50 vs. 60) so they don't disproportionately influence distance-based models like Logistic Regression.

3. Better Generalization: By "normalizing" the spread, the model focuses on broader trends rather than being distracted by the wide range of the "long tail."

---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [96]:
# 1. Derive Attributes based on domain knowledge and EDA insights

# Academic Efficiency: Ratio of passing (approved units aka passing grades) to effort (evaluated units aka taken exams)
# Note: Adding a small constant (1e-9) to the denominator to prevent division by zero, which can occur if a student has 0 evaluations.
df_clean['Academic_Efficiency_Ratio'] = df_clean['Curricular units 1st sem (approved)'] / (df_clean['Curricular units 1st sem (evaluations)'] + 1e-9)

# Financial Risk Score: Combined debtor and tuition fees 
# Note: Inverting 'Tuition fees' so that 1=Paid, 0=Debt, is to align the directionality of the risk score (Higher = More Risk).
df_clean['Financial_Risk_Score'] = df_clean['Debtor'] + (1 - df_clean['Tuition fees up to date'])

# Academic Momentum: Total progress in the first year (approved units across both semesters)
df_clean['Total_Academic_Momentum'] = df_clean['Curricular units 1st sem (approved)'] + df_clean['Curricular units 2nd sem (approved)']

# Grade Velocity: Performance change (grade difference between Sem 2 and Sem 1)
# Note: Negative = Performance Decay (risk signal), Positive = Performance Improvement (protective signal)
df_clean['Grade_Velocity_Trend'] = df_clean['Curricular units 2nd sem (grade)'] - df_clean['Curricular units 1st sem (grade)']

# --------------- Binary Flag (but necessary for recovery signal feature) --------------------
# Zero-Failure Flag: Captures the 0.0 grade spike from the histograms
df_clean['is_zero_sem1'] = (df_clean['Curricular units 1st sem (grade)'] == 0).astype(int)

# Recovery Signal: Specifically flags students who failed Sem 1 but passed Sem 2
# Note: This helps the model distinguish between 'Consistent' and 'Improving' students
df_clean['Recovery_Signal'] = df_clean['Grade_Velocity_Trend'] * df_clean['is_zero_sem1']




In [97]:
# --- 2. Binning / Binary Collapse ---

# Marital Status: Single (1) vs. All others (Riskier) (based on grouped countplot in EDA)
df_clean['is_single'] = (df_clean['Marital status'] == 1).astype(int)

# Previous Qualification: Standard Secondary (1) vs. Others (simplified decision boundary based on barchart)
df_clean['is_standard_secondary'] = (df_clean['Previous qualification'] == 1).astype(int)

# Credited Units: 0 vs. Any (based on boxplot & histogram: raw count has insufficient variance for most models)
df_clean['has_credits'] = (df_clean['Curricular units 1st sem (credited)'] > 0).astype(int)

# Maturity-Entry Interaction: Flagging those over age 25 who applied through specific application modes, based on grouped boxplot.
# Note: Application Mode 8 = 'Over 23 years old', and 39 = 'Over 23 (Special phase)'
df_clean['is_mature_entry'] = ((df_clean['Age at enrollment'] > 25) & (df_clean['Application mode'].isin([8, 39]))).astype(int)

# Late Switcher Flag: Change of Course (Application Mode 15) after 22 years old.
# Note: Based on grouped boxplot showing this mode had a higher median age and higher dropout rate.
df_clean['late_switcher'] = ((df_clean['Age at enrollment'] > 22) & (df_clean['Application mode'] == 15)).astype(int)

In [98]:
# ------------------3.1 Target Encoding --------------------------
# Convert complex categories into a single numeric "Risk Score" based on the probability of dropping out.

# Temporary mapping for risk calculation: 
# Dropout (High Risk) = 1, Enrolled (Medium) = 0.5, Graduate (Low) = 0
risk_mapping = {'Dropout': 1.0, 'Enrolled': 0.5, 'Graduate': 0.0}
temp_target_for_risk = df['Target'].map(risk_mapping)

high_card_features = ["Mother's occupation", "Father's occupation", "Application mode"]
scaler = RobustScaler()

for col in high_card_features:
    if col in df.columns:
        # Calculate the mean 'Risk' per category (groupby)
        encoding_map = temp_target_for_risk.groupby(df[col]).mean()
        
        # Create the new risk feature name
        new_col_name = f"{col.replace(' ', '_')}_risk"
        
        # Map risk scores and fill rare categories with the global mean risk
        df_clean[new_col_name] = df[col].map(encoding_map).fillna(temp_target_for_risk.mean())
        
        # Scale the new feature to match our normalized ranges
        df_clean[new_col_name] = scaler.fit_transform(df_clean[[new_col_name]])

# Print the mappings for deployment purposes
for col in high_card_features:
    if col in df.columns:
        mapping = temp_target_for_risk.groupby(df[col]).mean().to_dict()
        print(f"{col} mappings:")
        print(mapping)

Mother's occupation mappings:
{1: 0.6909722222222222, 2: 0.45588235294117646, 3: 0.44339622641509435, 4: 0.3831908831908832, 5: 0.3935128518971848, 6: 0.38301886792452833, 7: 0.3626373626373626, 8: 0.38235294117647056, 9: 0.5138888888888888, 10: 0.3944197844007609, 11: 0.5, 12: 0.7285714285714285, 13: 0.8235294117647058, 14: 0.5, 15: 0.42857142857142855, 16: 0.0, 17: 0.5, 18: 0.3333333333333333, 19: 0.5, 20: 0.25, 21: 0.3333333333333333, 22: 0.16666666666666666, 23: 0.16666666666666666, 24: 0.25, 25: 0.0, 26: 0.0, 27: 0.5, 28: 0.4, 29: 0.21153846153846154, 30: 0.3, 31: 0.25, 32: 0.4090909090909091}
Father's occupation mappings:
{1: 0.65234375, 2: 0.44402985074626866, 3: 0.48223350253807107, 4: 0.37890625, 5: 0.45595854922279794, 6: 0.3905038759689923, 7: 0.384297520661157, 8: 0.35960960960960964, 9: 0.38207547169811323, 10: 0.4004950495049505, 11: 0.42105263157894735, 12: 0.7076923076923077, 13: 0.7368421052631579, 14: 0.0, 15: 0.25, 16: 0.375, 17: 0.75, 18: 0.0, 19: 0.5, 20: 0.25, 21:

In [99]:
# 3.2 Label Encoding for the Target 
# Convert the 'Target' strings into integers for the model to predict.

le = LabelEncoder()
df_clean['Target'] = le.fit_transform(df_clean['Target'])

print("Target Mapping Key")
for index, label in enumerate(le.classes_):
    print(f"Class {index} -> {label}")

Target Mapping Key
Class 0 -> Dropout
Class 1 -> Enrolled
Class 2 -> Graduate


In [100]:
# --------------------- 4. Scaling ------------------
# RobustScaler handles the outliers we capped/log-transformed earlier
# This isnt necessary for tree-based models (e.g. XGBoost) but is important for linear models (e.g. Logistic Regression) due to outlier influence.

# Select continuous numerical indicators and engineered ratios for scaling (ensures that features with larger ranges
# do not numerically outweigh smaller ranges in linear models, while RobustScaler minimizes the impact of students
# with outlier academic patterns.


scaler = RobustScaler()
numerical_to_scale = [
    'Curricular units 1st sem (enrolled)', 
    'Curricular units 1st sem (evaluations)', 
    'Curricular units 1st sem (approved)', 
    'Curricular units 1st sem (grade)',
    'Academic_Efficiency_Ratio',
    'Total_Academic_Momentum',
    'Grade_Velocity_Trend',
    'Recovery_Signal'
]

df_clean[numerical_to_scale] = RobustScaler().fit_transform(df_clean[numerical_to_scale])

In [101]:
# Final pruning
drop_col = [
    'Unemployment rate', 'Inflation rate', 'GDP', 'Debtor', 
    'Nacionality', 'International', 'Educational special needs',
    'Curricular units 1st sem without evaluations', 'Application mode',
    'Curricular units 2nd sem without evaluations', 'Tuition fees up to date',
    'Curricular units 1st sem credited', 'Previous qualification',
    'Curricular units 2nd sem credited', 'Marital status', 'Age at enrollment',
    'Curricular units 2nd sem enrolled', "Father's occupation", 
    'Curricular units 2nd sem evaluations', "Mother's occupation",
    'Curricular units 2nd sem grade', "Father's qualification", 
    'Curricular units 2nd sem approved', "Mother's qualification", 
]

# Execute the drop on df_clean
df_clean = df_clean.drop(columns=[c for c in drop_col if c in df_clean.columns])

print(f"Cleanup Complete! Final Feature Count: {len(df_clean.columns)}")
df_clean.head()

Cleanup Complete! Final Feature Count: 34


,Application order,Course,Daytime/evening attendance,Displaced,Gender,Scholarship holder,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Target,Age_at_Enrollment_Log,Academic_Efficiency_Ratio,Financial_Risk_Score,Total_Academic_Momentum,Grade_Velocity_Trend,is_zero_sem1,Recovery_Signal,is_single,is_standard_secondary,has_credits,is_mature_entry,late_switcher,Mother's_occupation_risk,Father's_occupation_risk,Application_mode_risk
0,5,2,1,1,1,0,0,-2.0,-2.00,-1.666667,-5.464135,0,0,0,0,0,0.000000,0,0,3.044522,-1.121795,0,-1.666667,0.000000,1,0.0,1,1,0,0,0,-1.015319,0.000000,0.000000
1,1,11,1,1,1,0,0,0.0,-0.50,0.333333,0.738397,0,0,6,6,6,13.666667,0,2,2.995732,0.673077,1,0.333333,-0.305427,0,-0.0,1,1,0,0,0,-1.000000,-0.512234,-0.258403
2,5,5,1,1,1,0,0,0.0,-2.00,-1.666667,-5.464135,0,0,6,0,0,0.000000,0,0,2.995732,-1.121795,1,-1.666667,0.000000,1,0.0,1,1,0,0,0,0.000000,0.000000,-0.477134
3,2,15,1,1,0,0,0,0.0,0.00,0.333333,0.485232,0,0,6,10,5,12.400000,0,2,3.044522,0.224359,0,0.166667,-0.942460,0,-0.0,1,1,0,0,0,-1.015319,-0.512234,0.000000
4,1,3,0,0,0,0,0,0.0,0.25,0.000000,0.000000,0,0,6,6,6,13.000000,0,2,3.828641,-0.124644,0,0.166667,0.610854,0,0.0,0,1,0,0,0,0.000000,0.000000,1.238695


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [102]:
# TODO: Integrate data from multiple sources (if applicable).

# Example: Merge two DataFrames on a common key
# df_secondary = pd.read_csv('path/to/secondary_data.csv')
# df_integrated = pd.merge(df_clean, df_secondary, on='common_key', how='left')

# Example: Concatenate DataFrames with the same structure
# df_integrated = pd.concat([df_part1, df_part2], ignore_index=True)

# If using a single source, simply continue:
# df_integrated = df_clean.copy()
# print(f"Integrated dataset shape: {df_integrated.shape}")

In [103]:
# Optional: Verify the integrated data
# df_integrated.head()

This dataset served as a single source for the entire project. No external datasets were merged or integrated. Although web scraping was considered for potential feature enrichment, it was not implemented due to the adequacy of the existing dataset and project scope constraints. 
Model generalization was evaluated in phase 4, and despite a small difference, performance on unseen data remains strong, suggesting the model is stable and not severely overfitted (discussed further in report). As a result, the pipeline operates on a unified dataset with preprocessing and feature engineering.


---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [104]:
# --- 1. Data Type Conversions ---
# The binary flags (is_...) should be integers, and scaled values should be floats (this is already the 
# case for the engineered features, but we still perform a safety check to ensure all data types are correct before modeling).
binary_cols = [col for col in df_clean.columns if col.startswith('is_') or col == 'has_credits']
df_clean[binary_cols] = df_clean[binary_cols].astype(int)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[numeric_cols] = df_clean[numeric_cols].astype(float)

In [105]:
# --- 2. Column Reordering (Target Last) ---
target_col = 'Target'
feature_cols = [col for col in df_clean.columns if col != target_col]
df_final_ready = df_clean[feature_cols + [target_col]].copy()

# 3.Note: Column names are sufficient and descriptive, so we can skip renaming.

In [109]:
# 4. Save the prepared dataset for use in Phase 4 (Modelling).
OUTPUT_PATH = r'../data\processed\dataset_prepared_final.csv'
df_final_ready.to_csv(OUTPUT_PATH, index=False)
print(f"SUCCESS: Prepared dataset saved to: {OUTPUT_PATH}")
df_final_ready.head()

SUCCESS: Prepared dataset saved to: ../data\processed\dataset_prepared_final.csv


,Application order,Course,Daytime/evening attendance,Displaced,Gender,Scholarship holder,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Age_at_Enrollment_Log,Academic_Efficiency_Ratio,Financial_Risk_Score,Total_Academic_Momentum,Grade_Velocity_Trend,is_zero_sem1,Recovery_Signal,is_single,is_standard_secondary,has_credits,is_mature_entry,late_switcher,Mother's_occupation_risk,Father's_occupation_risk,Application_mode_risk,Target
0,5.0,2.0,1.0,1.0,1.0,0.0,0.0,-2.0,-2.00,-1.666667,-5.464135,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,3.044522,-1.121795,0.0,-1.666667,0.000000,1.0,0.0,1.0,1.0,0.0,0.0,0.0,-1.015319,0.000000,0.000000,0.0
1,1.0,11.0,1.0,1.0,1.0,0.0,0.0,0.0,-0.50,0.333333,0.738397,0.0,0.0,6.0,6.0,6.0,13.666667,0.0,2.995732,0.673077,1.0,0.333333,-0.305427,0.0,-0.0,1.0,1.0,0.0,0.0,0.0,-1.000000,-0.512234,-0.258403,2.0
2,5.0,5.0,1.0,1.0,1.0,0.0,0.0,0.0,-2.00,-1.666667,-5.464135,0.0,0.0,6.0,0.0,0.0,0.000000,0.0,2.995732,-1.121795,1.0,-1.666667,0.000000,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.000000,0.000000,-0.477134,0.0
3,2.0,15.0,1.0,1.0,0.0,0.0,0.0,0.0,0.00,0.333333,0.485232,0.0,0.0,6.0,10.0,5.0,12.400000,0.0,3.044522,0.224359,0.0,0.166667,-0.942460,0.0,-0.0,1.0,1.0,0.0,0.0,0.0,-1.015319,-0.512234,0.000000,2.0
4,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.000000,0.000000,0.0,0.0,6.0,6.0,6.0,13.000000,0.0,3.828641,-0.124644,0.0,0.166667,0.610854,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.000000,0.000000,1.238695,2.0


In [108]:
phase_4_modeling_strategy = {
    "1. Feature-Signal Finalization": {
        "Observation": "Engineered features like 'Grade_Velocity_Trend' and 'Academic_Efficiency' now provide normalized performance metrics.",
        "Action": "Use the final 34 features as the primary predictors, ensuring 'Target' is isolated as the dependent variable."
    },
    "2. Model Selection & Comparison": {
        "Finding": "Tabular student data involves complex, non-linear relationships between demographics and grades.",
        "Action": "Train and compare Random Forest and XGBoost classifiers to determine which handles multi-class variance better."
    },
    "3. Hyperparameter Optimization": {
        "Observation": "Standard model defaults may lead to overfitting on deep trees or underperforming on minority classes.",
        "Action": "Implement RandomizedSearchCV to tune 'n_estimators', 'max_depth', and 'learning_rate' for optimal F1-scores."
    },
    "4. Rigorous Evaluation Suite": {
        "Observation": "Accuracy alone is misleading in multi-class problems with imbalance.",
        "Action": "Deploy a AUC-ROC value, Classification Report, and Confusion Matrix to audit Precision and Recall for each specific student status."
    },
    "5. Pipeline Export & Deployment": {
        "Observation": "The processed dataset is now staged in 'data/processed/student_dropout_prepared_final.csv'.",
        "Action": "Ensure the finalized model is saved (using Joblib or Pickle) for potential integration into a real-time student monitoring dashboard."
    }
}

print("PHASE 4: STRATEGIC MODELING PLAN")
for key, content in phase_4_modeling_strategy.items():
    print(f"\n{key.upper()}")
    for k, v in content.items():
        print(f"  > {k}: {v}")

PHASE 4: STRATEGIC MODELING PLAN

1. FEATURE-SIGNAL FINALIZATION
  > Observation: Engineered features like 'Grade_Velocity_Trend' and 'Academic_Efficiency' now provide normalized performance metrics.
  > Action: Use the final 34 features as the primary predictors, ensuring 'Target' is isolated as the dependent variable.

2. MODEL SELECTION & COMPARISON
  > Finding: Tabular student data involves complex, non-linear relationships between demographics and grades.
  > Action: Train and compare Random Forest and XGBoost classifiers to determine which handles multi-class variance better.

3. HYPERPARAMETER OPTIMIZATION
  > Observation: Standard model defaults may lead to overfitting on deep trees or underperforming on minority classes.
  > Action: Implement RandomizedSearchCV to tune 'n_estimators', 'max_depth', and 'learning_rate' for optimal F1-scores.

4. RIGOROUS EVALUATION SUITE
  > Observation: Accuracy alone is misleading in multi-class problems with imbalance.
  > Action: Deploy a AU